# Day 68: Lab - Digit Classification with Keras

## Project Goal
This lab is a small end-to-end classification project: given an 8x8 grayscale image of a handwritten digit, predict which digit (0-9) it is. We use scikit-learn's built-in `load_digits` dataset instead of the full MNIST dataset - it's the same kind of task but smaller (1,797 images of 8x8 pixels) and ships with scikit-learn, so it works offline with no download. The code below loads the data, scales the pixel values with `StandardScaler` (neural networks train more reliably when input features are on a similar, small-value scale), and one-hot encodes the labels with `to_categorical` since we'll use a softmax output with categorical crossentropy loss. It also splits the data into train and test sets so we can measure performance on digits the model never saw during training.</cell id="cell-1">


In [ ]:
# pip install tensorflow
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical

digits = load_digits()
X, y = digits.data, digits.target  # X: flattened 8x8 images (64 features), y: digit labels 0-9
X = StandardScaler().fit_transform(X)  # scale pixel values to zero mean / unit variance for stable training
y_cat = to_categorical(y)  # one-hot encode labels (e.g. 3 -> [0,0,0,1,0,0,0,0,0,0]) for softmax + categorical_crossentropy
X_train, X_test, y_train, y_test = train_test_split(X, y_cat, test_size=0.2, random_state=42)  # 80/20 train/test split

## Step 1: Visualize a Few Digits
Before building any model, it's worth actually looking at the data - this catches obvious problems (wrong labels, corrupted images, unexpected formats) that are much harder to spot once everything is turned into numbers. Here we plot the first 5 digit images alongside their true labels using the original unflattened `digits.images` (8x8 grids), which is easier to visualize than the flattened 64-length vectors used for training. This is a standard first step in any image classification project: confirm the images look like what you expect and the labels line up correctly before spending time training a model.</cell id="cell-3">


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(digits.images[i], cmap='gray')  # raw 8x8 image, not the scaled/flattened version used for training
    ax.set_title(digits.target[i])  # true label shown above each image
    ax.axis('off')
plt.show()

## Step 2: Build & Train the Network
Since these images are tiny (8x8 = 64 pixels) and the patterns are simple compared to real-world photos, a fully-connected (Dense) network is enough - we don't need a CNN here. The architecture takes the 64 flattened pixel values as input, passes them through a 64-unit and then a 32-unit ReLU hidden layer to learn combinations of pixel patterns, uses Dropout(0.2) to reduce overfitting, and ends with a 10-unit softmax layer that outputs a probability distribution over the 10 possible digits. `categorical_crossentropy` is the correct loss for multi-class classification with one-hot encoded labels, and `validation_split=0.2` reserves part of the training data to monitor generalization while training runs for 30 epochs.</cell id="cell-5">


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(64,)),  # input: 64 flattened pixel values per image
    layers.Dropout(0.2),  # randomly drops 20% of activations during training to reduce overfitting
    layers.Dense(32, activation='relu'),
    layers.Dense(10, activation='softmax')  # 10 output probabilities, one per digit class (0-9)
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])  # categorical_crossentropy for multi-class, one-hot labels
model.fit(X_train, y_train, epochs=30, validation_split=0.2, verbose=0)

## Step 3: Evaluate
Validation accuracy during training gives a running estimate of generalization, but the real test is performance on the completely held-out test set (`X_test`, `y_test`) that was never touched during training or validation. `model.evaluate` runs a forward pass over the test set and returns the loss and any tracked metrics - here, test accuracy, which is the fraction of test images the model classified correctly. This final number is what you'd report as the model's real-world performance, since it reflects how the model behaves on data it has genuinely never seen.</cell id="cell-7">


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)  # loss and accuracy on unseen test data
print('Test accuracy:', test_acc)